In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from tqdm import tqdm # Para ver el progreso de las 20 pasadas

from monai.inferers import sliding_window_inference
from monai import transforms
from monai.networks.nets import SwinUNETR

# Importar tus clases unificadas
from src.get_data import UnifiedDataset
from src.custom_transforms import (
    ImputeMissingChannelsd,
    ConvertToMultiChannelPipeline2_Experimento_d
)

# Comando mágico de Jupyter
%matplotlib inline

# 1. SETUP DE DATOS Y MODELO
UPENN_DIR = "./Dataset/Dataset_30_6/"
MUGLIOMA_DIR = "./Dataset/MU_glioma/"
PIPELINE = 2
MODEL_WEIGHTS_PATH = "./Dataset_Output/pipe2/resilient-glade-12/model_best.pt"
roi = (128, 128, 64)
ID_BUSCADO = "UPENN-GBM-00307"
# ID_BUSCADO = "PatientID_0170"

val_transform = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"]),
    transforms.EnsureChannelFirstd(keys=["image", "label"]),
    ImputeMissingChannelsd(keys=["image"]),
    ConvertToMultiChannelPipeline2_Experimento_d(keys=["label"]),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
test_set = UnifiedDataset(upenn_dir=UPENN_DIR, muglioma_dir=MUGLIOMA_DIR, section="val", pipeline=PIPELINE, transform=val_transform)

model = SwinUNETR(img_size=roi, in_channels=11, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
if os.path.exists(MODEL_WEIGHTS_PATH):
    checkpoint = torch.load(MODEL_WEIGHTS_PATH, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    print("Pesos P2 cargados.")

# --- FUNCIÓN CLAVE PARA ACTIVAR E INYECTAR DROPOUT ---
def enable_dropout(m, prob=0.2):
    for each_module in m.modules():
        if each_module.__class__.__name__.startswith('Dropout'):
            each_module.train() # Despertamos la capa

            # EL TRUCO: Si la probabilidad estaba en 0.0, la forzamos a 0.2 (20%)
            if hasattr(each_module, 'p'):
                each_module.p = prob

# 2. BÚSQUEDA E INFERENCIAS
indice_encontrado = next((i for i, d in enumerate(test_set.data) if ID_BUSCADO in d["label"]), None)

if indice_encontrado is not None:
    datos = test_set[indice_encontrado]
    imagen = datos["image"].unsqueeze(0).to(device)
    etiqueta = datos["label"].unsqueeze(0).to(device)

    # --- PASADA 1: Inferencia Estándar (Determinista) ---
    print("Calculando Inferencia Estándar...")
    model.eval() # Modo evaluación estricto (Dropout apagado)
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            logits_std = sliding_window_inference(imagen, roi, sw_batch_size=1, predictor=model, overlap=0.5)
        # prob_std_np = torch.sigmoid(logits_std).squeeze(0).cpu().numpy()

        # --- APLICAMOS TEMPERATURE SCALING ---
        temperatura_optima = 1.0 # Empieza probando con 2.0 o 2.5
        logits_escalados = logits_std / temperatura_optima
        prob_std_np = torch.sigmoid(logits_escalados).squeeze(0).cpu().numpy()

        torch.cuda.empty_cache()

    # --- PASADA 2: Monte Carlo Dropout (20 iteraciones) ---
    print("Calculando Monte Carlo Dropout (20 pasadas)...")
    n_simulaciones = 2
    predicciones_mc = []

    model.eval()
    enable_dropout(model, prob=0.2) # Forzamos un 20% de neuronas apagadas

    with torch.no_grad():
        for _ in tqdm(range(n_simulaciones)):
            with torch.cuda.amp.autocast():
                logits_mc = sliding_window_inference(imagen, roi, sw_batch_size=1, predictor=model, overlap=0.5)
            # Guardamos la probabilidad en la RAM (cpu) para no reventar la VRAM
            prob_mc = torch.sigmoid(logits_mc).squeeze(0).cpu()
            predicciones_mc.append(prob_mc)
            del logits_mc
            torch.cuda.empty_cache()

    # Calculamos Mean (Probabilidad Calibrada) y Variance (Incertidumbre)
    tensor_mc = torch.stack(predicciones_mc) # [20, 2, H, W, D]
    prob_mc_mean_np = torch.mean(tensor_mc, dim=0).numpy()
    prob_mc_var_np = torch.var(tensor_mc, dim=0).numpy()

    # Preparamos las imágenes base
    img_np = imagen.squeeze(0).cpu().numpy()
    gt_np = etiqueta.squeeze(0).cpu().numpy()
    max_z = img_np.shape[3] - 1
    corte_z_optimo = int(np.argmax(np.sum(gt_np[0], axis=(0, 1))))

    # 3. FUNCIÓN DE VISUALIZACIÓN INTERACTIVA (Grilla 2x5)
    def plot_slices(corte_z):
        fig, axes = plt.subplots(2, 5, figsize=(26, 10))
        fig.suptitle(f'Evaluación Probabilística - {ID_BUSCADO} | Corte Z: {corte_z}', fontsize=18, fontweight='bold')

        # ==========================================
        # FILA 0: CANAL 0 - INFILTRACIÓN PURA
        # ==========================================
        bg0 = img_np[9, :, :, corte_z] # T1GD
        gt0 = np.ma.masked_where(gt_np[0, :, :, corte_z] == 0, gt_np[0, :, :, corte_z])
        p_std_0 = np.ma.masked_where(prob_std_np[0, :, :, corte_z] < 0.05, prob_std_np[0, :, :, corte_z])
        p_mean_0 = np.ma.masked_where(prob_mc_mean_np[0, :, :, corte_z] < 0.05, prob_mc_mean_np[0, :, :, corte_z])
        p_var_0 = np.ma.masked_where(prob_mc_var_np[0, :, :, corte_z] < 0.005, prob_mc_var_np[0, :, :, corte_z])

        # 0. Base
        axes[0, 0].imshow(bg0, cmap="gray"); axes[0, 0].set_title('T1GD Base'); axes[0, 0].axis('off')
        # 1. Ground Truth
        axes[0, 1].imshow(bg0, cmap="gray"); axes[0, 1].imshow(gt0, cmap="Reds", alpha=0.6)
        axes[0, 1].set_title('GT: Infiltración'); axes[0, 1].axis('off')
        # 2. STD Prob
        axes[0, 2].imshow(bg0, cmap="gray"); im_s0 = axes[0, 2].imshow(p_std_0, cmap="hot", alpha=0.7, vmin=0, vmax=1)
        axes[0, 2].set_title('Prob. Estándar (Sobreconfianza)'); axes[0, 2].axis('off'); fig.colorbar(im_s0, ax=axes[0, 2], fraction=0.046, pad=0.04)
        # 3. MCD Mean
        axes[0, 3].imshow(bg0, cmap="gray"); im_m0 = axes[0, 3].imshow(p_mean_0, cmap="hot", alpha=0.7, vmin=0, vmax=1)
        axes[0, 3].set_title('MCD Mean (Calibrada)'); axes[0, 3].axis('off'); fig.colorbar(im_m0, ax=axes[0, 3], fraction=0.046, pad=0.04)
        # 4. MCD Variance
        axes[0, 4].imshow(bg0, cmap="gray"); im_v0 = axes[0, 4].imshow(p_var_0, cmap="magma", alpha=0.8, vmin=0)
        axes[0, 4].set_title('MCD Varianza (Incertidumbre)'); axes[0, 4].axis('off'); fig.colorbar(im_v0, ax=axes[0, 4], fraction=0.046, pad=0.04)

        # ==========================================
        # FILA 1: CANAL 1 - TUMOR CORE ANCLA
        # ==========================================
        bg1 = img_np[7, :, :, corte_z] # FLAIR
        gt1 = np.ma.masked_where(gt_np[1, :, :, corte_z] == 0, gt_np[1, :, :, corte_z])
        p_std_1 = np.ma.masked_where(prob_std_np[1, :, :, corte_z] < 0.05, prob_std_np[1, :, :, corte_z])
        p_mean_1 = np.ma.masked_where(prob_mc_mean_np[1, :, :, corte_z] < 0.05, prob_mc_mean_np[1, :, :, corte_z])
        p_var_1 = np.ma.masked_where(prob_mc_var_np[1, :, :, corte_z] < 0.005, prob_mc_var_np[1, :, :, corte_z])

        axes[1, 0].imshow(bg1, cmap="gray"); axes[1, 0].set_title('FLAIR Base'); axes[1, 0].axis('off')

        axes[1, 1].imshow(bg1, cmap="gray"); axes[1, 1].imshow(gt1, cmap="Blues", alpha=0.6)
        axes[1, 1].set_title('GT: Tumor Core'); axes[1, 1].axis('off')

        axes[1, 2].imshow(bg1, cmap="gray"); im_s1 = axes[1, 2].imshow(p_std_1, cmap="winter", alpha=0.7, vmin=0, vmax=1)
        axes[1, 2].set_title('Prob. Estándar'); axes[1, 2].axis('off'); fig.colorbar(im_s1, ax=axes[1, 2], fraction=0.046, pad=0.04)

        axes[1, 3].imshow(bg1, cmap="gray"); im_m1 = axes[1, 3].imshow(p_mean_1, cmap="winter", alpha=0.7, vmin=0, vmax=1)
        axes[1, 3].set_title('MCD Mean'); axes[1, 3].axis('off'); fig.colorbar(im_m1, ax=axes[1, 3], fraction=0.046, pad=0.04)

        axes[1, 4].imshow(bg1, cmap="gray"); im_v1 = axes[1, 4].imshow(p_var_1, cmap="magma", alpha=0.8, vmin=0)
        axes[1, 4].set_title('MCD Varianza (Incertidumbre)'); axes[1, 4].axis('off'); fig.colorbar(im_v1, ax=axes[1, 4], fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

    # 4. CREAR EL WIDGET INTERACTIVO
    slider = widgets.IntSlider(min=0, max=max_z, step=1, value=corte_z_optimo, description='Corte Z:')
    widgets.interact(plot_slices, corte_z=slider)

else:
    print(f"ID {ID_BUSCADO} no encontrado en el dataset.")

[VAL] Cargados 6 casos de UPenn-GBM (Pipeline 2)
[VAL] Cargados 21 casos de MU-Glioma Post (Pipeline 2)


/home/minigo/anaconda3/envs/monai_env/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.nets.swin_unetr SwinUNETR.__init__:img_size: Argument `img_size` has been deprecated since version 1.3. It will be removed in version 1.5. The img_size argument is not required anymore and checks on the input size are run during forward().
  warn_deprecated(argname, msg, warning_category)


Pesos P2 cargados.
Calculando Inferencia Estándar...
Calculando Monte Carlo Dropout (20 pasadas)...


100%|██████████| 2/2 [00:05<00:00,  2.91s/it]


interactive(children=(IntSlider(value=123, description='Corte Z:', max=154), Output()), _dom_classes=('widget-…